In [100]:
import sys
import os
from astropy.table import Table, unique, hstack
from astropy.table import join
from matplotlib import pyplot as plt

import numpy as np
import pandas as pd

from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u
import numpy as np

In [101]:
cat = 'glimpse_s07'
alma_file = 'almaprops_2024_all_20250219.tab'
alma_sources_file = '7MTM2TM1_Core_catalogue_out_official_v3_run6Apr23+16Jan24_sn-5_may24_clump_cat.txt'

alma_regions = Table.read("../alma_input/" + alma_file, format="ascii.ipac")
alma_sources = Table.from_pandas(
    pd.read_csv(
        "../alma_input/" + alma_sources_file,
        comment="\\",
        skiprows=[2],
        delimiter=r"\s+",
    )
)

alma_sources.rename_column("S/N", "S_N")
alma_sources.rename_column("FWHM_max/FWHM_min", "FWHM_max__FWHM_min")
alma_sources.rename_column("Bmaj/Bmin", "Bmaj_Bmin")

alma_sources.rename_column("CLUMP.1", "CLUMP_1")

alma_sources.rename_column("Dcore/BEAM_size", "Dcore__Beam_size")
alma_sources.rename_column("Lclump/Mclump", "Lclump_Mclump")
alma_sources.rename_column("n(H2)", "n_H2")

spitzer_sources = Table.read(
    "../output/tables/"+cat+"/spitzer_sources_" + cat + "_download.tbl",
    format="ipac",
)
spitzer_sources = unique(spitzer_sources, keys="designation")

In [102]:
alma_coords = SkyCoord(alma_sources["RA"], alma_sources["DEC"], unit="deg", frame="icrs")

spitzer_coords = SkyCoord(
    spitzer_sources["ra"], spitzer_sources["dec"], unit="deg", frame="icrs"
)

idx, sep, _ = spitzer_coords.match_to_catalog_sky(alma_coords)


radius = 0.6 * u.arcsec
mask = sep < radius

unique, counts = np.unique(mask,return_counts=True)
print(dict(zip(unique,counts)))

{np.False_: np.int64(98711), np.True_: np.int64(405)}


In [103]:
print(spitzer_sources[mask])
print(alma_sources[idx[mask]])
print(len(sep[mask]))

cntr_01   dist_x     pang_x   ... color45_58 color45_80 color58_80
          arcsec      deg     ...                                 
------- --------- ----------- ... ---------- ---------- ----------
      6 10.089022   98.640331 ...      0.647         --         --
      9  0.609209   80.476043 ...         --         --         --
     11  7.777152   -7.668469 ...      0.562      1.018      0.456
     13  4.297785   -49.96121 ...      0.708      1.287      0.579
     13   8.05396   33.247811 ...      1.014      1.882      0.868
     15  2.354947   68.382146 ...       1.62      2.718      1.098
     16   3.43762  138.939045 ...       0.63      0.429     -0.201
     16  2.336904   64.055499 ...      1.203      1.515      0.312
     17 10.424115   -87.72436 ...        1.3         --         --
    ...       ...         ... ...        ...        ...        ...
    994  5.132344  105.001185 ...      1.805         --         --
    997  1.642561  106.947732 ...      1.586      2.478      0

In [104]:
spitzer_sources['alma_match'] = mask

In [105]:
spitzer_sources['alma_region'] = alma_sources['CLUMP'][idx]
spitzer_sources['alma_id'] = alma_sources['ID'][idx]
spitzer_sources['alma_sep'] = sep

alma_sources['ALMA_MATCH'] = True


In [106]:
spitzer_sources[]

SyntaxError: invalid syntax (4096076920.py, line 1)

In [ ]:
alma_sources

ID,CLUMP,X,Y,RA,DEC,GLON,GLAT,F_PEAK,S_N,FWHM_X,FWHM_Y,FWHM_max__FWHM_min,FWHM_circ,Bmaj_Bmin,BEAM_circ,PA,F_INT,RMS_map,CLUMP_1,N_FRAG,DIST,Lclump,Mclump,Tclump,Surfd,Tcore,Mcore,Dcore,BEAM_size,Dcore__Beam_size,Lclump_Mclump,EVOL_FLAG,Dclump,Subsample,GAUSS_FLAG,Dcore_min,Dcore_max,Bmaj,Bmin,BPA,Back_onpeak,ID_ord,F_PEAK_jybm,Back_onpeak_jybm,Rcore,Surfd_core,n_H2,AG_ID,ORIG_ID,RADIO_INFO,RADIO_MATCH,Surfd_nd,VLSR_clump,ALMA_MATCH
int64,str18,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str18,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str4,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,str17,str22,int64,int64,float64,float64,bool
1,100132,897.03558,807.0044,277.80605,-8.9093544,22.706657,0.39130379,966.05285,24.27594482512637,0.3176051,0.2108376,1.5063968665930554,0.2587722879903487,1.5066741852215375,0.2533616015841717,25.371765,0.001724063,39.79465503645834,100132,2,6.05,1817.1,796.6,14.7,0.49,35.0,0.7560295217136781,1565.5723423416098,1532.8376895842391,1.0213555896882007,2.2810695455686667,2.0,0.66,far,1,1275.56748,1921.510855,0.310992896556864,0.206410184502588,-60.50614929199,90.593484,1,0.0016515686534744,0.0001548790611025,782.7861711708049,3.509695192129958,71500000.0,AG022.7068+0.3897,HIGALBM22.7066+0.3893,1,0,0.3,115.8,True
2,100132,796.61502,862.99532,277.80701,-8.9088256,22.707565,0.39070761,386.48921,9.712088461274844,0.30397822,0.20706381,1.468041276744594,0.2508842131147717,1.5066741852215375,0.2533616015841717,12.305599,0.0006483369,39.79465503645834,100132,2,6.05,1817.1,796.6,14.7,0.49,35.0,0.2843062210698383,1517.8494893443692,1532.8376895842391,0.9902219260775514,2.2810695455686667,2.0,0.66,far,1,1252.7360505,1839.068231,0.310992896556864,0.206410184502588,-60.50614929199,69.352815,2,0.000660743834193,0.0001185659100164,758.9247446721846,1.4041253248865404,29500000.0,AG022.7068+0.3897,HIGALBM22.7066+0.3893,1,0,0.3,115.8,True
4,100244,468.07833,931.00007,278.45735,-9.2166486,22.731143,-0.3210833,402.87712,8.800255644314067,0.1880314,0.29689158,1.5789468142023089,0.2362730188481367,1.527675867927155,0.2446371339067752,-56.278564,0.00059940114,45.780160972971615,100244,1,5.61,468.1,995.0,12.3,0.7,20.0,0.4463725477044001,1325.4916357380473,1372.414321217009,0.9658101167019648,0.4704522613065326,2.0,0.62,far,1,1054.8561539999998,1665.5617638,0.302369505167016,0.197927787899988,-62.39258575439,22.965,1,0.0006421426528582,3.660373173559397e-05,662.7458178690237,2.8908158112557216,69600000.0,AG022.7285-0.3194,HIGALBM22.7285-0.3196,1,0,0.43,104.6,True
1,100309,768.08734,459.99554,278.38876,-9.1760131,22.735978,-0.2422652,599.23798,11.977861940756336,0.26129859,0.31598157,1.2092739191589208,0.2873421979225924,1.4718282946705987,0.2677463554286418,282.8571,0.0013186054,50.02879336595204,100309,7,5.61,1398.9,1152.7,15.2,1.3,35.0,0.4971820559253654,1611.9897303457435,1502.0570539546804,1.0731880830369445,1.213585494924959,2.0,0.49,far,1,1465.8850899,1772.6566077,0.324827015399928,0.22069627046586,-60.42407226562,126.44644,1,0.0011440911367911,0.0002414170264755,805.9948651728719,2.177047133853787,43100000.0,AG022.7384-0.2420,HIGALBM22.7385-0.2421,1,0,0.65,104.6,True
12,100309,696.04825,692.00105,278.38951,-9.1736286,22.738435,-0.24182238,679.1169,13.574520877055267,0.33104418,0.23101322,1.4330096779742736,0.2765421884343501,1.4718282946705987,0.2677463554286418,175.57346,0.0013841528,50.02879336595204,100309,7,5.61,1398.9,1152.7,15.2,1.3,35.0,0.5218967970393957,1551.401677116704,1502.0570539546804,1.0328513640890715,1.213585494924959,2.0,0.49,far,1,1295.9841642,1857.1578498,0.324827015399928,0.22069627046586,-60.42407226562,818.18009,2,0.0012965994347272,0.0015621049074158,775.700838558352,2.467249363365609,50700000.0,AG022.7384-0.2420,HIGALBM22.7385-0.2421,1,0,0.65,104.6,True
16,100309,620.16646,729.99563,278.

In [107]:
catalogue = join(left = spitzer_sources,
                 right =alma_sources,
                 join_type= 'outer',
                 keys_left=['alma_region','alma_id','alma_match'],
                 keys_right=['CLUMP','ID','ALMA_MATCH']
)

catalogue.remove_columns(['ALMA_MATCH'])

In [108]:
catalogue

cntr_01,dist_x,pang_x,internal_id_01,ra_01,dec_01,designation,l,b,ra,dec,clon,clat,dra,ddec,csf,mag_j,mag_h,mag_ks,mag3_6,d3_6m,mag4_5,d4_5m,mag5_8,d5_8m,mag8_0,d8_0m,f_j,f_h,f_ks,f3_6,df3_6,f4_5,df4_5,f5_8,df5_8,f8_0,df8_0,color36_45,color36_58,color36_80,color45_58,color45_80,color58_80,alma_match,alma_region,alma_id,alma_sep,ID,CLUMP,X,Y,RA,DEC,GLON,GLAT,F_PEAK,S_N,FWHM_X,FWHM_Y,FWHM_max__FWHM_min,FWHM_circ,Bmaj_Bmin,BEAM_circ,PA,F_INT,RMS_map,CLUMP_1,N_FRAG,DIST,Lclump,Mclump,Tclump,Surfd,Tcore,Mcore,Dcore,BEAM_size,Dcore__Beam_size,Lclump_Mclump,EVOL_FLAG,Dclump,Subsample,GAUSS_FLAG,Dcore_min,Dcore_max,Bmaj,Bmin,BPA,Back_onpeak,ID_ord,F_PEAK_jybm,Back_onpeak_jybm,Rcore,Surfd_core,n_H2,AG_ID,ORIG_ID,RADIO_INFO,RADIO_MATCH,Surfd_nd,VLSR_clump
,arcsec,deg,,,,,deg,deg,deg,deg,,,arcsec,arcsec,,mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,,,,,,,,,,deg,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
int64,float64,float64,str18,float64,float64,str25,float64,float64,float64,float64,str12,str13,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,str18,int64,float64,int64,str18,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str18,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str4,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,str17,str22,int64,int64,float64,float64
128,46.95203,-153.119608,100132,277.80780029,-8.9104003906,SSTGLMC G022.6934+00.3891,22.693489,0.389133,277.801831,-8.922033,18h31m12.44s,-08d55m19.32s,0.3,0.3,0,--,--,--,13.829,0.14,13.41,0.173,--,--,--,--,--,--,--,0.826,0.1064,0.7774,0.124,--,--,--,--,0.419,--,--,--,--,--,False,100132,1,0.01334613501037222,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--
128,8.458113,-160.086867,100132,277.80780029,-8.9104003906,SSTGLMC G022.7042+00.3889,22.704201,0.388975,277.80699,-8.912609,18h31m13.68s,-08d54m45.39s,0.3,0.3,0,--,--,--,13.953,0.133,13.196,0.192,--,--,--,--,--,--,--,0.7366,0.09017,0.947,0.1671,--,--,--,--,0.757,--,--,--,--,--,False,100132,1,0.0033844970265777545,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--
128,53.169957,-74.400282,100132,277.80780029,-8.9104003906,SSTGLMC G022.7034+00.4037,22.70347,0.403736,277.793401,-8.906428,18h31m10.42s,-08d54m23.14s,0.3,0.3,0,13.109,12.781,12.635,12.603,0.072,12.453,0.109,12.189,0.195,11.757,0.17,9.097,7.905,5.887,2.554,0.169,1.877,0.1884,1.532,0.2748,1.272,0.1993,0.15,0.414,0.846,0.264,0.696,0.432,False,100132,1,0.012834511955308128,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--
128,41.475203,-78.662714,100132,277.80780029,-8.9104003906,SSTGLMC G022.7033+00.4003,22.703312,0.40035,277.796366,-8.908135,18h31m11.13s,-08d54m29.29s,0.3,0.3,0,14.116,12.946,12.593,12.275,0.065,12.265,0.093,12.005,0.11,--,--,3.598,6.791,6.12,3.454,0.2081,2.231,0.1904,1.814,0.1841,--,--,0.01,0.27,--,0.26,--,--,False,100132,1,0.009644571816093435,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--,--
128,20.629791,-98.41936,100132,277.80780029,-8.9104003906,SSTGLMC G022.7031+00.3939,22.703164,0.393925,277.802062,-8.911239,18h31m12.49s,-08d54m40.46s,0.3,0.3,0,--,--,--,14.005,0.087,13.804,0.151,--,--,--,--,--,--,--,0.7021,0.05651,0.5407,0.07497,--,--,--,--,0.201,--,--,-

In [109]:
spitzer_sources

cntr_01,dist_x,pang_x,internal_id_01,ra_01,dec_01,designation,l,b,ra,dec,clon,clat,dra,ddec,csf,mag_j,mag_h,mag_ks,mag3_6,d3_6m,mag4_5,d4_5m,mag5_8,d5_8m,mag8_0,d8_0m,f_j,f_h,f_ks,f3_6,df3_6,f4_5,df4_5,f5_8,df5_8,f8_0,df8_0,color36_45,color36_58,color36_80,color45_58,color45_80,color58_80,alma_match,alma_region,alma_id,alma_sep
,arcsec,deg,,,,,deg,deg,deg,deg,,,arcsec,arcsec,,mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,mJy,,,,,,,,,,deg
int64,float64,float64,str18,float64,float64,str25,float64,float64,float64,float64,str12,str13,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,bool,str18,int64,float64
1,57.111899,-142.353229,46677,272.72900391,-20.546899796,SSTGLMC G010.1172-00.7685,10.117284,-0.768578,272.718651,-20.559461,18h10m52.48s,-20d33m34.06s,0.3,0.3,0,--,--,--,13.372,0.099,13.612001,0.183,--,--,--,--,--,--,--,1.258,0.1142,0.6453,0.1085,--,--,--,--,-0.24,--,--,--,--,--,False,46677,1,0.010434490851250915
1,59.110303,-131.242348,46677,272.72900391,-20.546899796,SSTGLMC G010.1175-00.7654,10.117526,-0.765414,272.715814,-20.557724,18h10m51.80s,-20d33m27.81s,0.3,0.3,0,14.884,12.254,11.026,10.185,0.044,10.3,0.056,10.020001,0.063,9.97,0.053,1.774,12.84,25.91,23.700001,0.962,13.63,0.7005,11.29,0.66,6.595,0.3208,-0.115,0.165,0.215,0.28,0.33,0.05,False,46677,1,0.01076457059824992
1,54.073549,-141.37869,46677,272.72900391,-20.546899796,SSTGLMC G010.1181-00.7684,10.118159,-0.768456,272.718987,-20.558635,18h10m52.56s,-20d33m31.09s,0.3,0.3,0,--,--,--,14.223,0.167,13.801,0.179,--,--,--,--,--,--,--,0.5745,0.08818,0.542,0.08918,--,--,--,--,0.422,--,--,--,--,--,False,46677,1,0.009564314379267856
1,52.87289,-148.563815,46677,272.72900391,-20.546899796,SSTGLMC G010.1182-00.7703,10.118289,-0.770342,272.720819,-20.559431,18h10m53.00s,-20d33m33.95s,0.3,0.3,1,--,13.089,12.307,11.767,0.072,11.776,0.07,11.586,0.114,12.38,0.297,--,5.952,7.964,5.518,0.3649,3.5,0.2248,2.67,0.2811,0.716,0.1962,-0.009,0.181,-0.613,0.19,-0.604,-0.794,False,46677,1,0.009536686956067122
1,52.775077,-163.159491,46677,272.72900391,-20.546899796,SSTGLMC G010.1186-00.7740,10.11862,-0.774056,272.724464,-20.560931,18h10m53.87s,-20d33m39.35s,0.3,0.3,0,12.42,11.423,11.07,10.742,0.067,10.65,0.083,10.694,0.055,10.721,0.067,17.16,27.610001,24.889999,14.18,0.8725,9.871,0.7584,6.07,0.3065,3.301,0.2024,0.092,0.048,0.021,-0.044,-0.071,-0.027,False,46677,1,0.010459993938960506
1,51.406589,-159.122631,46677,272.72900391,-20.546899796,SSTGLMC G010.1188-00.7729,10.118817,-0.772986,272.723565,-20.560242,18h10m53.66s,-20d33m36.87s,0.3,0.3,0,--,--,--,13.496,0.098,13.142,0.122,--,--,--,--,--,--,--,1.122,0.1011,0.9952,0.1116,--,--,--,--,0.354,--,--,--,--,--,False,46677,1,0.00979487119068753
1,50.409492,-147.909554,46677,272.72900391,-20.546899796,SSTGLMC G010.1189-00.7702,10.118981,-0.770214,272.721055,-20.558763,18h10m53.05s,-20d33m31.55s,0.3,0.3,1,--,--,--,12.974,0.118,12.661,0.109,--,--,--,--,--,--,--,1.814,0.197,1.549,0.1557,--,--,--,--,0.313,--,--,--,--,--,False,46677,1,0.008833438945804963
1,52.821383,-130.774319,46677,272.72900391,-20.546899796,SSTGLMC G010.1192-00.7659,10.119209,-0.765897,272.717133,-20.556482,18h10m52.11s,-20d33m23.34s,0.3,0.3,0,15.112001,14.172,13.762,13.465,0.105,13.371,0.17,--,--,--,--,1.438,2.195,2.085,1.155,0.1121,0.8057,0.1261,--,--,--,--,0.094,--,--,--,--,--,False,46677,1,0.009015568855245503


In [110]:
catalogue.write('../output/jae/catalogue.ecsv',format='ascii.ecsv',overwrite=True)
catalogue.write('../output/jae/catalogue.fits',format='fits',overwrite=True)

In [112]:
catalogue2 = Table.read('../output/jae/catalogue.fits',format='fits')

In [54]:
print(len(spitzer_sources)+len(alma_sources)-len(mask[mask]))

105059



spitzer_sources_mask = spitzer_sources[mask]
#spitzer_sources['merge_idx'] = np.zeros(len(spitzer_sources))-1
spitzer_sources_mask['merge_idx'] = np.array(range(0,len(mask[mask]),1))

spitzer_sources_merged = join(spitzer_sources,
                       spitzer_sources_mask['designation','merge_idx'],
                       keys_left='designation',
                       keys_right='designation',
                       join_type= 'left',
                       uniq_col_name= '{col_name})

alma_sources_mask = alma_sources[idx[mask]]
alma_sources_mask['merge_idx'] = np.array(range(0,len(idx[mask]),1))

#try hstack

alma_sources_merged = join(alma_sources,
                       alma_sources_mask['ID','CLUMP'],
                       keys_left=['ID','CLUMP'],
                       keys_right =['ID','CLUMP'],
                       join_type= 'left')

In [ ]:
catalogue = join(alma_sources[idx[mask]],
                 spitzer_sources[mask],
                 join_type='left'
                 )

TableMergeError: No keys in common between left and right tables

In [30]:
with open('../output/jae/compare_match_astropy.csv','w') as file:

    for line in spitzer_sources[mask]['designation']:
        file.write(str(line))
        file.write('\n')





In [35]:
match_table = Table.read('../output/tables/match_table_glimpse_s07.tbl',format='ipac')

In [37]:
with open('../output/jae/compare_match_mine.csv','w') as file:

    for line in match_table['spitzer_source']:
        file.write(str(line))
        file.write('\n')



In [56]:
list1 = spitzer_sources[mask]['designation']
list2 = match_table['spitzer_source']

In [60]:
list_discrepancies = list(set(list1) ^ set(list2))
print(len(list_discrepancies))

with open('../output/jae/compare_match_discrepancies.csv','w') as file:

    for line in list_discrepancies:
        file.write(str(line))
        file.write('\n')

print(len(list_discrepancies))

17
17


In [57]:
print(set(list1))

{np.str_('SSTGLMC G328.8542+00.3069'), np.str_('SSTGLMC G345.0042-00.2236'), np.str_('SSTGLMC G024.5411+00.3121'), np.str_('SSTGLMC G028.5817-00.3405'), np.str_('SSTGLMC G019.0766-00.2884'), np.str_('SSTGLMC G309.3374-00.6429'), np.str_('SSTGLMC G309.9118+00.3249'), np.str_('SSTGLMC G028.3260+00.1611'), np.str_('SSTGLMC G332.2805-00.5471'), np.str_('SSTGLMC G345.3341+01.0196'), np.str_('SSTGLMC G031.1009+00.2651'), np.str_('SSTGLMC G310.6055+00.5465'), np.str_('SSTGLMC G331.2779-00.1883'), np.str_('SSTGLMC G348.5772-00.9209'), np.str_('SSTGLMC G028.3581+00.0743'), np.str_('SSTGLMC G333.0679-00.4468'), np.str_('SSTGLMC G011.9357-00.6133'), np.str_('SSTGLMC G311.6418+00.2061'), np.str_('SSTGLMC G309.2357-00.4558'), np.str_('SSTGLMC G024.4424-00.2277'), np.str_('SSTGLMC G341.2380-00.2701'), np.str_('SSTGLMC G333.9300-00.1317'), np.str_('SSTGLMC G284.3389-00.3871'), np.str_('SSTGLMC G327.1647-00.2495'), np.str_('SSTGLMC G016.5080-00.4392'), np.str_('SSTGLMC G328.3192+00.3974'), np.str_('SS

In [ ]:
alma_sources_

ID,CLUMP,X,Y,RA,DEC,GLON,GLAT,F_PEAK,S_N,FWHM_X,FWHM_Y,FWHM_max__FWHM_min,FWHM_circ,Bmaj_Bmin,BEAM_circ,PA,F_INT,RMS_map,CLUMP_1,N_FRAG,DIST,Lclump,Mclump,Tclump,Surfd,Tcore,Mcore,Dcore,BEAM_size,Dcore__Beam_size,Lclump_Mclump,EVOL_FLAG,Dclump,Subsample,GAUSS_FLAG,Dcore_min,Dcore_max,Bmaj,Bmin,BPA,Back_onpeak,ID_ord,F_PEAK_jybm,Back_onpeak_jybm,Rcore,Surfd_core,n_H2,AG_ID,ORIG_ID,RADIO_INFO,RADIO_MATCH,Surfd_nd,VLSR_clump
int64,str18,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str18,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str4,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,str17,str22,int64,int64,float64,float64
12,49005,282.84716,398.01379,272.49237,-19.811085,10.670233,-0.22136326,114.90239,8.155696955718524,0.39615926,0.4616445,1.165300288575862,0.4276502583923807,1.1738058713494082,0.4517980359991473,44.392556,0.00056004538,14.088604643338783,49005,5,3.53,606.8,622.4,12.4,1.77,20.0,0.1651303190020806,1509.605412125104,1594.8470670769902,0.9465518313877482,0.9749357326478147,2.0,0.3,near,1,1398.4421878,1629.605085,0.48948848247528,0.4170097410678,65.36275482178,46.968813,3,0.0006246430231325,0.0002553362148974,754.802706062552,0.8244737991326049,17400000.0,AG010.6691-0.2196,HIGALBM10.6693-0.2188,1,0,0.89,30.4
5,G010.8856+00.1221,420.17799,416.97852,272.28317,-19.456649,10.885094,0.12256048,1423.5503,63.105776649377226,0.55744977,0.56750627,1.018040190419309,0.5624555446300249,1.145792179943996,0.4159114606498355,45.32226,0.012002339,22.558161480357104,G010.8856+00.1221,4,2.64,4486.5,88.8,24.4,1.59,50.0,0.6690459626616406,1484.8826378232657,1098.0062561155655,1.352344423861808,50.52364864864865,0.0,0.0,near,1,1471.6673927999998,1498.2165528,0.44519862532608,0.38855093717592,68.14502716064,140.88738,2,0.0065582617592691,0.0006490647479176,742.4413189116328,3.4526187370021364,74200000.0,AG010.8851+0.1225,HIGALBM10.8852+0.1231,1,0,1.59,20.0
13,50555,327.13979,412.00488,272.99201,-19.596941,11.084598,-0.53027883,535.03591,49.91065611859404,0.63642167,0.53658939,1.1860496719847553,0.5843775455029748,1.2077646826500557,0.5456711546172215,6.690877,0.0048695258,10.71987330177922,50555,6,3.57,307.8,970.4,12.0,3.32,20.0,1.4685113028870185,2086.22783744562,1948.0460219834808,1.0709335477205226,0.3171887881286068,2.0,0.26,near,1,1915.6241223,2272.0253619,0.59968358278272,0.49652352929124,87.35166168213,26.490981,6,0.0042428636830942,0.0002100749110736,1043.11391872281,3.839111703677257,58700000.0,AG011.0829-0.5311,HIGALBM11.0827-0.5316,1,0,1.49,32.0
4,51554,913.50649,978.94471,271.8988,-18.709561,11.362044,0.80272408,447.96762,8.321409253393446,0.26272596,0.16198125,1.621952911216576,0.2062927032355967,1.585226600875286,0.2007104410288609,-5.6074195,0.00050807879,53.83314368504592,51554,5,3.19,99.5,192.6,13.1,2.5,20.0,0.1223394776096217,658.0737233215536,640.2663068820665,1.027812515273847,0.5166147455867082,2.0,0.25,far,1,516.7201875,838.0958123999999,0.25270605087282,0.1594132035970679,82.51621246338,-5.3151598,4,0.000480618748113,-5.70256718353141e-06,329.0368616607768,3.214359512630259,156000000.0,AG011.3614+0.8017,HIGALBM11.3618+0.8016,1,0,0.53,27.4
6,51554,672.47028,1120.958,271.90085,-18.708417,11.363988,0.80158358,506.36666,9.406224963612177,0.23671524,0.1528949,1.548221948541122,0.1902434044803551,1.585226600875286,0.2007104410288609,-179.81217,0.00048842854,53.83314368504592,51554,5,3.19,99.5,192.6,13.1,2.5,20.0,0.1176079254031254,606.8764602923328,640.2663068820665,0.9478500645265348,0.5166147455867082,2.0,0.25,far,1,487.734731,755.1216155999999,0.25270605087282,0.1594132035970679,82.51621246338,87.25531,5,0.0005432743335676,9.361510963317792e-05,303.4382301461664,3.633397657088857,191000000.0,AG011.3614+0.8017,HIGALBM11.3618+0.8016,1,0,0.53,27.4
12,53795,246.93773,265.01437,273.04766,-18.691371,11.903988,-0.141118

In [42]:
spitzer_sources_merged['merge_idx']

--
--
--
--
--
--
--
--
--
--
--


In [34]:
print(len(idx))
print(len(spitzer_sources))

99116
99116
